# 04 - Incident Timeline và Root Cause

Notebook cuối kết hợp evidence từ metrics và logs để trả lời đầy đủ:

- **WHEN**: anomaly bắt đầu từ lúc nào?
- **WHERE**: service nào là nơi lỗi chính?
- **WHAT**: cơ chế/root cause hypothesis là gì?

Notebook này tự đọc raw data từ `g2-data/g2`, tự tính lại timeline, không phụ thuộc vào file CSV trong `lab/results`.

In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "g2-data" / "g2").exists():
            return candidate
    raise FileNotFoundError("Cannot find project root containing g2-data/g2")

ROOT = find_project_root()
DATA = ROOT / "g2-data" / "g2"
METRICS = DATA / "metrics"
LOGS = DATA / "logs"
PLOTS = ROOT / "lab" / "plots"
PLOTS.mkdir(parents=True, exist_ok=True)

def format_time_axis(ax, interval=3):
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=interval))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    ax.tick_params(axis="x", labelrotation=0, labelsize=8, labelbottom=True)

def load_metric(name):
    df = pd.read_csv(METRICS / f"{name}.csv")
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    return df

def first_sustained(df, series, threshold, points=5):
    mask = series >= threshold
    sustained = mask.rolling(points).sum() >= points
    if not sustained.any():
        return None
    idx = np.where(sustained)[0][0] - points + 1
    return df.loc[idx, "timestamp"], float(series.iloc[idx])

def first_log_timestamp(path, needle):
    with open(path, encoding="utf-8") as f:
        for line in f:
            event = json.loads(line)
            if needle in event["message"]:
                return pd.to_datetime(event["timestamp"], utc=True)
    return None

def first_30m_spike(path, needle, min_count=100):
    timestamps = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            event = json.loads(line)
            if needle in event["message"]:
                timestamps.append(pd.to_datetime(event["timestamp"], utc=True))
    counts = pd.Series(1, index=pd.DatetimeIndex(timestamps)).resample("30min").size()
    high = counts[counts >= min_count]
    return high.index.min() if not high.empty else None

cart = load_metric("cart-service")
product = load_metric("product-service")
gateway = load_metric("api-gateway")
order = load_metric("order-service")
payment = load_metric("payment-service")

cart["memory_pct"] = 100 * cart["memory_usage_bytes"] / cart["memory_limit_bytes"]
cart["restart_delta"] = cart["container_restart_count"].diff().fillna(0)

## 4.1 Tự dựng lại các mốc từ raw metrics/logs

In [ ]:
# Product early suspicious signal
product_p99_ts, _ = first_sustained(product, product["http_p99_latency_ms"], 100)
product_5xx_ts, _ = first_sustained(product, product["http_5xx_rate"], 5)

# Cart metrics
cart_gc_ts, _ = first_sustained(cart, cart["jvm_gc_pause_ms_avg"], 100)
cart_mem_ts, _ = first_sustained(cart, cart["memory_pct"], 70)

# Cart multivariate anomaly via Isolation Forest
feature_cols = ["memory_pct", "jvm_gc_pause_ms_avg", "http_p99_latency_ms", "http_5xx_rate", "restart_delta"]
features = cart[feature_cols].astype(float).ffill().bfill()
scaled = StandardScaler().fit_transform(features)
model = IsolationForest(n_estimators=300, contamination=0.04, random_state=42)
pred = model.fit_predict(scaled)
first_if_idx = np.where(pred == -1)[0][0]
cart_if_ts = cart.loc[first_if_idx, "timestamp"]

# Logs
cart_log = LOGS / "cart-service.log.jsonl"
cache_first_ts = first_log_timestamp(cart_log, "ProductCatalogCache eviction failed")
cache_spike_ts = first_30m_spike(cart_log, "ProductCatalogCache eviction failed", min_count=100)
oom_imminent_ts = first_log_timestamp(cart_log, "OutOfMemoryError imminent")
oomkilled_ts = first_log_timestamp(cart_log, "Container OOMKilled")

# Restart
restart_rows = cart[cart["restart_delta"] > 0]
restart_ts = restart_rows["timestamp"].iloc[0]

# Downstream metrics
gateway_ts, _ = first_sustained(gateway, gateway["cart_upstream_error_rate"], 5)
order_ts, _ = first_sustained(order, order["upstream_timeout_rate"], 10)
payment_ts, _ = first_sustained(payment, payment["upstream_timeout_rate"], 10)

timeline = pd.DataFrame(
    [
        (product_p99_ts, "product-service", "Product p99 latency sustained >100 ms"),
        (product_5xx_ts, "product-service", "Product 5xx sustained >5%"),
        (cache_first_ts, "cart-service", "ProductCatalogCache eviction failures first appear"),
        (cache_spike_ts, "cart-service", "Cache eviction failures become high-volume"),
        (cart_if_ts, "cart-service", "Isolation Forest first multivariate anomaly"),
        (cart_gc_ts, "cart-service", "GC pause sustained >100 ms"),
        (cart_mem_ts, "cart-service", "Memory sustained >70% of limit"),
        (oom_imminent_ts, "cart-service", "OutOfMemoryError imminent"),
        (oomkilled_ts, "cart-service", "Container OOMKilled"),
        (restart_ts, "cart-service", "First container restart recorded"),
        (gateway_ts, "api-gateway", "Cart upstream error rate sustained >5%"),
        (order_ts, "order-service", "Order upstream timeout rate sustained >10%"),
        (payment_ts, "payment-service", "Payment upstream timeout rate sustained >10%"),
    ],
    columns=["timestamp_dt", "service", "evidence"],
).sort_values("timestamp_dt")

timeline["timestamp"] = timeline["timestamp_dt"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
timeline[["timestamp", "service", "evidence"]]

## 4.2 Cross-signal timeline

In [ ]:
service_y = {name: idx for idx, name in enumerate(timeline["service"].unique())}
timeline["y"] = timeline["service"].map(service_y)

fig, ax = plt.subplots(figsize=(13, 6))
ax.scatter(timeline["timestamp_dt"], timeline["y"], s=80, color="#1f77b4")
for _, row in timeline.iterrows():
    ax.text(row["timestamp_dt"], row["y"] + 0.08, row["evidence"], fontsize=7, rotation=25, ha="left")
ax.set_yticks(list(service_y.values()), list(service_y.keys()))
ax.set_title("Cross-signal incident timeline")
ax.grid(True, axis="x", alpha=0.25)
format_time_axis(ax)
fig.tight_layout()
fig.savefig(PLOTS / "04_incident_cross_signal_timeline.png", dpi=160)
plt.show()

**Cross-signal incident timeline**

![Cross-signal incident timeline](../plots/04_incident_cross_signal_timeline.png)

## 4.3 Xếp hạng service theo first anomaly

In [ ]:
first_anomaly = pd.DataFrame(
    [
        ("product-service", product_p99_ts, "early suspicious signal"),
        ("cart-service", cache_first_ts, "first proven cart signal"),
        ("api-gateway", gateway_ts, "cart upstream errors"),
        ("order-service", order_ts, "upstream timeouts"),
        ("payment-service", payment_ts, "upstream timeouts"),
    ],
    columns=["service", "timestamp", "meaning"],
)
first_anomaly["timestamp_dt"] = pd.to_datetime(first_anomaly["timestamp"], utc=True)
first_anomaly["timestamp"] = first_anomaly["timestamp_dt"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
first_anomaly

In [ ]:
start = first_anomaly["timestamp_dt"].min()
first_anomaly["hours_since_first"] = (first_anomaly["timestamp_dt"] - start).dt.total_seconds() / 3600

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(first_anomaly["service"], first_anomaly["hours_since_first"], color="#4c78a8")
ax.set_xlabel("giờ sau signal đầu tiên")
ax.set_title("Xếp hạng first anomaly theo service")
ax.grid(True, axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(PLOTS / "04_service_first_anomaly_ranking.png", dpi=160)
plt.show()

**Xếp hạng first anomaly theo service**

![Xếp hạng first anomaly theo service](../plots/04_service_first_anomaly_ranking.png)

## 4.4 Kết luận WHEN / WHERE / WHAT

**WHEN**

- Earliest suspicious signal: `product-service` lúc `03:03Z`.
- Earliest proven cart signal: `ProductCatalogCache eviction failed` lúc `06:32:33Z`.
- First high-volume cart signal: template spike lúc `14:00Z`.
- First strong metric anomaly trên cart: Isolation Forest lúc `18:04:30Z`, GC sustained lúc `18:06Z`.
- Failure thật sự: OOM/OOMKilled lúc `19:59Z`, restart đầu tiên lúc `20:00Z`.

**WHERE**

- Root failure nằm ở `cart-service`.
- `api-gateway`, `order-service`, và `payment-service` là downstream symptoms vì chúng bất thường sau cart.

**WHAT**

- Giả thuyết root cause: `cart-service` bị memory pressure liên quan tới `ProductCatalogCache`.
- Cache eviction failures xuất hiện sớm và spike từ `14:00Z`.
- Heap pressure làm GC pause tăng, memory tăng, cuối cùng OOMKilled và restart loop.
- Product-service có anomaly rất sớm, có thể là possible trigger, nhưng chưa đủ evidence để gọi là root cause.

## 4.5 Limitations

- Không có product-service logs, nên chưa chứng minh trực tiếp product anomaly gây ra cart cache issue.
- Không có distributed traces để nối quan hệ request/cache refresh giữa product và cart.
- Metrics thiếu 30 phút từ `11:30Z` đến `11:59:30Z`.
- Rolling Z-score có thể bắt outlier sớm, nên phải kết hợp sustained threshold và Isolation Forest.